# onto-canon6 Policy Contracts

This notebook is the first local PoC for `onto-canon6`.

It does one thing only: inspect how the new ontology-runtime contracts
resolve unknown ontology items under `open`, `closed`, and `mixed`
policies.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Could not locate onto-canon6 repo root from notebook cwd")

for candidate_path in (PROJECT_ROOT / "src", PROJECT_ROOT.parent / "llm_client"):
    if candidate_path.exists():
        candidate_str = str(candidate_path)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

from pprint import pprint



from onto_canon6.ontology_runtime import (  # noqa: E402
    OntologyPolicy,
    PackRef,
    UnknownItemPolicy,
    build_proposal_request,
    decide_unknown_item,
)

print(PROJECT_ROOT)


/home/brian/projects/onto-canon6


In [2]:
policies = {
    "open": OntologyPolicy(mode="open"),
    "closed": OntologyPolicy(mode="closed"),
    "mixed": OntologyPolicy(mode="mixed", proposal_policy="allow"),
    "closed_with_overlay_override": OntologyPolicy(
        mode="closed",
        proposal_policy="allow",
        overlay_target=PackRef(pack_id="project_local_overlay", pack_version="0.1.0"),
        unknown_items=UnknownItemPolicy(predicate="propose"),
    ),
}

pprint(policies)


{'closed': OntologyPolicy(mode='closed', proposal_policy='reject', unknown_items=UnknownItemPolicy(default_action=None, predicate=None, role=None, entity_type=None, value_kind=None), overlay_target=None),
 'closed_with_overlay_override': OntologyPolicy(mode='closed', proposal_policy='allow', unknown_items=UnknownItemPolicy(default_action=None, predicate='propose', role=None, entity_type=None, value_kind=None), overlay_target=PackRef(pack_id='project_local_overlay', pack_version='0.1.0')),
 'mixed': OntologyPolicy(mode='mixed', proposal_policy='allow', unknown_items=UnknownItemPolicy(default_action=None, predicate=None, role=None, entity_type=None, value_kind=None), overlay_target=None),
 'open': OntologyPolicy(mode='open', proposal_policy='reject', unknown_items=UnknownItemPolicy(default_action=None, predicate=None, role=None, entity_type=None, value_kind=None), overlay_target=None)}


In [3]:
decision_rows = []
for name, policy in policies.items():
    decision = decide_unknown_item(
        policy=policy,
        kind="predicate",
        value="oc:unknown_predicate_demo",
    )
    decision_rows.append(
        {
            "policy": name,
            "action": decision.action,
            "reason": decision.reason,
            "target_pack": decision.target_pack.model_dump() if decision.target_pack else None,
        }
    )

pprint(decision_rows)


[{'action': 'allow',
  'policy': 'open',
  'reason': "default ontology mode 'open' resolved action 'allow' for "
            'predicate',
  'target_pack': None},
 {'action': 'reject',
  'policy': 'closed',
  'reason': "default ontology mode 'closed' resolved action 'reject' for "
            'predicate',
  'target_pack': None},
 {'action': 'propose',
  'policy': 'mixed',
  'reason': "default ontology mode 'mixed' resolved action 'propose' for "
            'predicate',
  'target_pack': None},
 {'action': 'propose',
  'policy': 'closed_with_overlay_override',
  'reason': "explicit unknown-item policy resolved action 'propose' for "
            'predicate',
  'target_pack': {'pack_id': 'project_local_overlay', 'pack_version': '0.1.0'}}]


In [4]:
proposal_decision = decide_unknown_item(
    policy=policies["closed_with_overlay_override"],
    kind="predicate",
    value="oc:unknown_predicate_demo",
)

proposal_request = build_proposal_request(proposal_decision)
pprint(proposal_request.model_dump())


{'kind': 'predicate',
 'reason': "explicit unknown-item policy resolved action 'propose' for "
           'predicate',
 'target_pack': {'pack_id': 'project_local_overlay', 'pack_version': '0.1.0'},
 'value': 'oc:unknown_predicate_demo'}
